# 03 - Time-ordered split and baseline model

**Purpose:** set up an evaluation that mirrors how a fraud model is used in production (a model trains on past data and scored on the future), then fit a simple logistic regression to give a comparison model a target to beat.

In [1]:
# Imports
import json
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

In [2]:
# Same load-and-join as the notebooks 01 and 02
tx = pd.read_csv("../data/train_transaction.csv")
idn = pd.read_csv("../data/train_identity.csv")
df = tx.merge(idn, on='TransactionID', how='left')
print(df.shape)

(590540, 434)


In [3]:
# Helper columns
# TransactionDT is seconds from an arbitrary start point, so convert to days for the split
df['day'] = df['TransactionDT'] / 86400

#  Missing identity data is informative so keep as 0/1 flag
df['has_identity'] = df['TransactionID'].isin(idn['TransactionID']).astype(int)

# As mentioned in EDA add log transformations for transaction amount
df['log_amt'] = np.log1p(df['TransactionAmt'])
df['log_amt_sq'] = df['log_amt'] ** 2

del tx, idn # free up memory

In [4]:
# Load the feature list from 02 EDA notebook
with open("../reports/features.json") as f:
    features = json.load(f)
numeric_features, cat_cols = features['numeric'], features['categorical']
all_features = numeric_features + cat_cols

# Keep only the columes we need for efficient memory usage
df = df[all_features + ['isFraud', 'TransactionDT', 'day']]
print(f"{len(numeric_features)} numeric + {len(cat_cols)} categorical features")

386 numeric + 25 categorical features


In [5]:
# Split by time not at random. In production the model is trained on past transactions and scores future ones. A random split would let the model learn from the future and overstate performance.
df = df.sort_values('TransactionDT')

train = df[df['day'] < 120]                         # fit the model
valid = df[(df['day'] >= 120) & (df['day'] < 150)]  # make decisions
test = df[df['day'] >= 150]                         # held out

for name, part in [('train', train), ('valid', valid), ('test', test)]:
    print(f"{name:5s} rows = {len(part):>7,}, fraud = {part['isFraud'].sum():>5,}, "
        f"fraud rate = {part['isFraud'].mean():.2%}")


train rows = 410,601, fraud = 14,419, fraud rate = 3.51%
valid rows =  85,303, fraud = 2,962, fraud rate = 3.47%
test  rows =  94,636, fraud = 3,282, fraud rate = 3.47%


Train covers days 0-119, validation days 120-149, and test days 150 onwards (roughly a month). Fraud rate by split: 3.51%, 3.47%, 3.47% - the rates are similar across periods.
The test set is not scored in this notebook, as if I tuned on it, my final result would be optimistic.

In [6]:
# Build the pipeline
# Numeric columns: median imputation then scaling.
# Median is robust to skewed values. However, it hides missingness, which the has_identity flag only partly offsets.
# Scaling puts features on comparable ranges so the optimiser converges.
numeric_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical columns: missing becomes its own category, then one-hot encode.
# This keeps the 'was missing' signal. handle_unknown='ignore' means a category that appears only after training produces all zeros instead of an error.
cat_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# ColumnTransformer applies each recipe to its own set of columns
preprocess = ColumnTransformer(transformers=[
    ('num', numeric_pipe, numeric_features),
    ('cat', cat_pipe, cat_cols)
])

# Full pipeline: every processing step is fitted on train only
baseline = Pipeline(steps=[
    ('prep', preprocess),
    ('model', LogisticRegression(max_iter=1000))
])

In [7]:
# Fit the model to the training data
baseline.fit(train[all_features], train['isFraud'])

n_cols = baseline.named_steps['prep'].get_feature_names_out().shape[0]
print('Model input columns after encoding:', n_cols)

Model input columns after encoding: 472


In [8]:
# Evaluate baseline model on validation data
# predict_proba gives each transaction a probability of being fraud
p_valid = baseline.predict_proba(valid[all_features])[:, 1]
y_valid = valid['isFraud']

# ROC AUC: how well the model ranks fraud above genuine overall (0.5 = random)
roc = roc_auc_score(y_valid, p_valid)

# PR AUC: more informative when fraud is rare. A random model scores the base fraud rate.
pr = average_precision_score(y_valid, p_valid)

print('ROC AUC:', round(roc, 4))
print('PR AUC: ', round(pr, 4), '| random model:', round(y_valid.mean(), 4))

# To be practical, we review only the top 1% of transactions by score
# Precision = share of reviewed transactions that are fraud
# Recall = share of all fraud that the review catches
n_review = int(0.01 * len(valid))
top = valid.assign(score=p_valid).nlargest(n_review, 'score')
precision_top = top['isFraud'].mean()
recall_top = top['isFraud'].sum() / y_valid.sum()
print(f"Top 1% reviewed ({n_review:,} transactions): " f"precision={precision_top:.1%}, recall={recall_top:.1%}")

ROC AUC: 0.8537
PR AUC:  0.4258 | random model: 0.0347
Top 1% reviewed (853 transactions): precision=79.0%, recall=22.8%


In [9]:
# Save the baseline's validation results so the challenger notebook can compare against exactly these results
metrics = {
    'model': 'logistic regression model',
    'roc_auc_valid': round(float(roc), 4),
    'pr_auc_valid': round(float(pr), 4),
    'precision_top1pct': round(float(precision_top), 4),
    'recall_top1pct': round(float(recall_top), 4)
}

with open ("../reports/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)

{'model': 'logistic regression model', 'roc_auc_valid': 0.8537, 'pr_auc_valid': 0.4258, 'precision_top1pct': 0.7902, 'recall_top1pct': 0.2275}


## What I found

- **Baseline perfomance:** ROC AUC (0.8536), PR AUC (0.4257) against a random-model PR AUC (0.0347)
- **Top 1% review:** precision (0.7878), recall (0.2269). Roughly 4 in 5 reviewed transactions are fraud, about 22x the validation fraud rate of 3.47%. Recall is 22.7%, meaning about a fifth of all fraud is caught. However, recall is capped by review volume, at 1% capacity, the maximum possible is about 28.8% (1% / 3.47%), so the baseline captures roughly 79% of what any model could at this volume.
- **Same split everywhere:** the model is trained on days 0-119 and judged on a later period, which is the setting it would face in production.

## Limitations
- **Linear model:** logistic regression can't capture interactions between features or most non-linear effects.
- **Median imputation:** hides missingness for numeric columns, partly offset by 'has_identity'.
- **Coarse feature list:** columns with more than 90% missing, text columns with more than 20 categories and ID-like codes were dropped for scopre (see notebook 02). The baseline is therefore a deliberately simple reference point and not a tuned model.

## Next steps
1. Fit a gradient-boosted comparison model on the same split and the same features, and compar it with this baseline on validation.
2. Check whether the improvement justifies the extra complexity.